# 01 | Dataset understanding and data dictionary

**Goal:** identify the cases, layouts, plan families, source fields and target before modelling. Data: UrbanTALES idealized LES profiles and catalogue metadata (Nazarian et al., 2025).

The data have three nested levels, and the validation design in notebook 04 depends on them:

| Level | Count | Meaning |
|---|---:|---|
| case (`simulation_id`) | 224 | one LES run = one layout at one wind direction (0° or 45°) |
| layout (`layout_id`) | 112 | one building arrangement; its 0° and 45° runs |
| **plan family (`family_id`)** | **16** | one footprint (aligned/staggered) at one plan-area density; holds 7 sibling layouts that differ only in height pattern |

Field definitions are in `docs/data_dictionary.csv`; extraction steps in `docs/dataset_extraction.md`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'; RESULTS = ROOT / 'results'
import sys; sys.path.insert(0, str(ROOT / 'src'))
df = pd.read_csv(DATA / 'layout_level_dataset.csv')
target = 'target_U_canopy_mean_m_s'
print(json.dumps(json.loads((DATA/'dataset_summary.json').read_text()), indent=2)[:1800])
print('\nFirst rows:'); print(df.head(4).to_string(index=False))

{
  "source": "UrbanTALES idealized building arrays; profile-*.csv and metadata_ideal.json",
  "cases": 224,
  "layout_groups": 112,
  "target": "Height-weighted mean of profile U (horizontally averaged wind speed) from z=0 to mean building height for each simulation case",
  "target_units": "m s-1",
  "profile_cases": 224,
  "target_summary": {
    "count": 224.0,
    "mean": 2.0822327547481803,
    "std": 1.0023046172583712,
    "min": 0.9659274491936395,
    "25%": 1.3902258627976192,
    "50%": 1.629748004625278,
    "75%": 2.471943818184731,
    "max": 5.275317173872631
  },
  "grouping": "NameI identifies base geometry; d00 and d45 directions from a base layout remain together in each grouped fold",
  "features": [
    "plan_area_density",
    "height_mean_m",
    "height_std_m",
    "height_max_m",
    "wind_direction_deg",
    "configuration",
    "horizontal_configuration",
    "density_class",
    "building_count"
  ],
  "validation": "5-fold GroupKFold using layout_id groups

## Hierarchy check

Every family must contain exactly 7 layouts (CL/HR/SM x 28/56, plus the uniform case) and 14 cases.

In [2]:
h = df.groupby('family_id').agg(layouts=('layout_id','nunique'), cases=('simulation_id','nunique'))
assert (h.layouts == 7).all() and (h.cases == 14).all() and len(h) == 16
print(h.T.to_string())
print('\nExample family A35:', sorted(df[df.family_id=='A35'].layout_id.unique()))

family_id  A0625  A1111  A16  A25  A35  A4444  A53  A64  S0625  S1111  S16  S25  S35  S4444  S53  S64
layouts        7      7    7    7    7      7    7    7      7      7    7    7    7      7    7    7
cases         14     14   14   14   14     14   14   14     14     14   14   14   14     14   14   14

Example family A35: ['UA35', 'VA35-CL-28', 'VA35-CL-56', 'VA35-HR-28', 'VA35-HR-56', 'VA35-SM-28', 'VA35-SM-56']


## Labelling convention (CL / SM)

The catalogue's own `Config` field disagrees with the folder code: folders coded `-CL-` carry the metadata label `idealized-SM` and vice versa (HR and uniform agree). The folder code is the stable identifier and is used in **every** file of this repository (`configuration`). The raw catalogue label is kept as `configuration_metadata_label` for traceability only and is never a predictor. This is a pure relabelling, so model accuracy is unaffected. We could not tell from the metadata alone which labelling the dataset authors intend; confirm with the dataset documentation before describing CL or SM physically.

In [3]:
print(pd.crosstab(df.configuration, df.configuration_metadata_label))

configuration_metadata_label  idealized-CL  idealized-HR  idealized-SM  idealized-uniform
configuration                                                                            
idealized-CL                             0             0            64                  0
idealized-HR                             0            64             0                  0
idealized-SM                            64             0             0                  0
idealized-uniform                        0             0             0                 32


## Data dictionary

In [4]:
dictionary = pd.read_csv(ROOT/'docs'/'data_dictionary.csv')
print(dictionary.to_string(index=False))

                   field    units    type                                                                 meaning                                  source                      model_use
           simulation_id     none  string                UrbanTALES simulation case identifier; never a predictor                     metadata_ideal.json grouping and traceability only
               layout_id     none  string                 Base geometry identifier shared by d00 and d45 variants     Folder Name before direction suffix       GroupKFold grouping only
target_U_canopy_mean_m_s      m/s   float Trapezoidal mean of supplied profile U from z=0 to mean building height profile-<case>.csv and metadata Heights                    Target only
       plan_area_density    ratio   float                            Plan area density for the idealized geometry                    metadata areaDensity              Numeric predictor
           height_mean_m        m   float                     Arithmetic me